# Exercise 1 – Visualising waiting times

**Goal:** show how the average waiting time develops per month (Sep 2023 – Sep 2024), including the spread (min–max).

**Chart choice:** an interactive line chart with a range band. The data describes *change over time*, so a line is the right form. Min and max are not separate series but the boundaries around the average, so they are shown as a shaded band instead of three separate lines or grouped bars. Hover over any month to see min, average and max.

In [1]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_excel('Waiting_times.xlsx')
df['Period'] = pd.to_datetime(df['Month'] + ' ' + df['Year'].astype(str), format='%b %Y')
df = df.sort_values('Period').reset_index(drop=True)
df['Label'] = df['Period'].dt.strftime('%b %Y')
df

,Year,Month,Min,Avg waiting time,Max,Period,Label
0,2023,Sep,9,18,27,2023-09-01,Sep 2023
1,2023,Oct,6,12,17,2023-10-01,Oct 2023
2,2023,Nov,7,13,17,2023-11-01,Nov 2023
3,2023,Dec,12,18,26,2023-12-01,Dec 2023
4,2024,Jan,12,17,24,2024-01-01,Jan 2024
5,2024,Feb,7,13,20,2024-02-01,Feb 2024
6,2024,Mar,7,12,18,2024-03-01,Mar 2024
7,2024,Apr,9,14,21,2024-04-01,Apr 2024
8,2024,May,8,13,21,2024-05-01,May 2024
9,2024,Jun,12,18,28,2024-06-01,Jun 2024


In [2]:
BLUE = '#2a78d6'
BAND = 'rgba(42,120,214,0.15)'
EDGE = 'rgba(42,120,214,0.35)'
INK, MUTED, AXIS = '#1f1f1d', '#6b6b66', '#d9d7d2'

avg, lo, hi = df['Avg waiting time'], df['Min'], df['Max']
mean_all = avg.mean()

fig = go.Figure()

# Min–max range band (max line first, then min filled up to it)
fig.add_trace(go.Scatter(x=df['Label'], y=hi, mode='lines', line=dict(color=EDGE, width=1),
                         name='Max', hovertemplate='Max: %{y}<extra></extra>', showlegend=False))
fig.add_trace(go.Scatter(x=df['Label'], y=lo, mode='lines', line=dict(color=EDGE, width=1),
                         fill='tonexty', fillcolor=BAND, name='Range (min–max)',
                         hovertemplate='Min: %{y}<extra></extra>'))

# Average line
fig.add_trace(go.Scatter(x=df['Label'], y=avg, mode='lines+markers', name='Average waiting time',
                         line=dict(color=BLUE, width=3),
                         marker=dict(size=9, color=BLUE, line=dict(color='white', width=2)),
                         hovertemplate='<b>Average: %{y}</b><extra></extra>'))

# Period average reference line
fig.add_hline(y=mean_all, line=dict(color=MUTED, dash='dash', width=1),
              annotation_text=f'Period average {mean_all:.1f}', annotation_position='bottom right',
              annotation_font=dict(color=MUTED, size=11))

# Selective labels
i_min, i_max, i_last = avg.idxmin(), avg.idxmax(), len(df) - 1
for i, txt, ay in [(i_min, f'Lowest: {avg[i_min]}', 28), (i_max, f'Peak: {avg[i_max]}', -28), (i_last, f'{avg[i_last]}', -24)]:
    fig.add_annotation(x=df['Label'][i], y=avg[i], text=f'<b>{txt}</b>', showarrow=False, yshift=-ay * 0.7,
                       font=dict(color=INK, size=12))
fig.add_annotation(x=df['Label'][i_max], y=hi[i_max], text=f'Max {hi[i_max]}', showarrow=False, yshift=12,
                   font=dict(color=MUTED, size=11))

fig.update_layout(
    title=dict(text='<b>Waiting times peaked in summer 2024</b><br>'
                    '<span style="font-size:13px;color:#6b6b66">Average waiting time per month with spread (min–max), Sep 2023 – Sep 2024</span>',
               x=0.02, font=dict(size=20, color=INK)),
    template='plotly_white', plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Inter, Segoe UI, Arial, sans-serif', color=MUTED),
    hovermode='x unified', hoverlabel=dict(bgcolor='white', font_color=INK),
    legend=dict(orientation='h', x=0, y=1.0, yanchor='bottom'),
    margin=dict(t=135, l=60, r=30, b=50), height=520,
    xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor=AXIS, ticks=''),
    yaxis=dict(showgrid=False, zeroline=False, showline=False, title='Waiting time',
               range=[0, hi.max() * 1.15]),
)

fig.write_html('waiting_times.html', include_plotlyjs=True)
fig.show()

**Interpretation:**
- From autumn 2023 to spring 2024 the average waiting time fluctuates around 12–14, but from June 2024 it rises clearly to a peak in **August 2024 (avg. 24, max 36)**.
- The spread also widens in summer: people not only wait longer on average, the outliers become more extreme as well.
- December/January show a smaller peak – possibly seasonal (holidays).
- Sep 2024 (21) is well above Sep 2023 (18): waiting times have increased year-on-year.